# 🗄️ Notebook 3: Database-Backed Deduplication (Exactly-Once Effect)

An in-memory map vanishes on restart and isn't shared across replicas. For real *exactly-once effect*, store the idempotency key inside the **same transaction** that performs the side effect, with a `UNIQUE` constraint to enforce uniqueness across all callers.

## 🧠 At-least-once vs at-most-once vs exactly-once

- **At-most-once**: send it and hope. Simple; may lose data. (email marketing)
- **At-least-once**: retry until acknowledged. May duplicate. (most messaging systems)
- **Exactly-once** (delivery) is usually impossible over a network — *but* you can get **exactly-once effect** by making the receiver idempotent. That's what we build here.


## 🛠️ Setup

```bash
cd 04-patterns/idempotency
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟩 SQLite implementation (no external services)

Both the balance update and the `INSERT` into `idem` happen inside one transaction. Either both commit, or neither does — no torn state.

In [ ]:
import sqlite3, uuid, json, time

conn = sqlite3.connect(':memory:', check_same_thread=False)
conn.executescript('''
CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER);
CREATE TABLE idem (
    key         TEXT PRIMARY KEY,
    result_json TEXT NOT NULL,
    created_at  REAL NOT NULL
);
INSERT INTO accounts VALUES ('alice', 100);
''')

def charge(key, account, amount):
    with conn:  # 'with conn' = BEGIN … COMMIT on success / ROLLBACK on error
        row = conn.execute('SELECT result_json FROM idem WHERE key=?', (key,)).fetchone()
        if row:
            return ('REPLAY', json.loads(row[0]))
        conn.execute('UPDATE accounts SET balance = balance - ? WHERE name=?', (amount, account))
        bal = conn.execute('SELECT balance FROM accounts WHERE name=?', (account,)).fetchone()[0]
        result = {'balance': bal, 'charged': amount}
        conn.execute(
            'INSERT INTO idem(key, result_json, created_at) VALUES (?, ?, ?)',
            (key, json.dumps(result), time.time()),
        )
        return ('FRESH', result)

k = str(uuid.uuid4())
for i in range(3):
    print(f'attempt {i+1}:', charge(k, 'alice', 10))
print('balance:', conn.execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0])


## 🧪 Why the transaction matters

Let's simulate a crash *between* the balance update and the idempotency insert, to show why they must be atomic.

In [ ]:
class FakeCrash(Exception):
    pass

def buggy_charge_no_tx(key, account, amount):
    # NOTE: no `with conn:` — each statement auto-commits as it runs.
    row = conn.execute('SELECT result_json FROM idem WHERE key=?', (key,)).fetchone()
    if row:
        return ('REPLAY', json.loads(row[0]))
    conn.execute('UPDATE accounts SET balance = balance - ? WHERE name=?', (amount, account))
    conn.commit()  # pretend the DB auto-committed — the money really moved
    raise FakeCrash('server died before writing the idempotency key!')
    conn.execute('INSERT INTO idem(key, result_json, created_at) VALUES (?, ?, ?)',
                 (key, json.dumps({}), time.time()))  # unreachable

before = conn.execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]
k2 = str(uuid.uuid4())
try:
    buggy_charge_no_tx(k2, 'alice', 10)
except FakeCrash as e:
    print('💥', e)
after = conn.execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]
print(f'balance before={before}, after={after}  ← money moved, but no idempotency record!')
print('On retry, the server has no record of the key → it would charge AGAIN.')


Now the same simulated crash with a proper transaction — SQLite rolls back the balance change:

In [ ]:
def crashing_charge_with_tx(key, account, amount):
    with conn:
        conn.execute('UPDATE accounts SET balance = balance - ? WHERE name=?', (amount, account))
        raise FakeCrash('server died — but inside a transaction')

before = conn.execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]
try:
    crashing_charge_with_tx(str(uuid.uuid4()), 'alice', 10)
except FakeCrash as e:
    print('💥', e, '→ transaction rolled back')
after = conn.execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]
print(f'balance before={before}, after={after}  ← unchanged, safe to retry ✅')


## 🧹 TTL: cleaning up old keys

Idempotency tables grow forever. After the retry window (e.g. 24 hours), old keys can be deleted. A client retrying after a week with an ancient key would then be treated as a *fresh* request — which is fine, because no reasonable client waits that long to retry.


In [ ]:
RETENTION_SECONDS = 24 * 3600  # 24h in production

def purge_old_idem_keys(now=None):
    now = now or time.time()
    cutoff = now - RETENTION_SECONDS
    with conn:
        cur = conn.execute('DELETE FROM idem WHERE created_at < ?', (cutoff,))
        return cur.rowcount

# Simulate an "old" key by inserting one with a back-dated timestamp:
with conn:
    conn.execute('INSERT INTO idem(key, result_json, created_at) VALUES (?, ?, ?)',
                 ('old-key', '{}', time.time() - 48 * 3600))
print('rows before purge:', conn.execute('SELECT COUNT(*) FROM idem').fetchone()[0])
print('purged:', purge_old_idem_keys())
print('rows after  purge:', conn.execute('SELECT COUNT(*) FROM idem').fetchone()[0])


## 🧠 Key takeaways

- Side-effect **and** the idempotency record must live in the **same transaction**.
- `UNIQUE`/`PRIMARY KEY` on the idempotency key gives you dedup for free across replicas.
- Keep keys only as long as clients may retry; purge the rest.
- In the real world, Stripe / AWS lock the key row while the request is in-flight and return `409 Conflict` if another request with the same key is still processing — see notebook 4 for that concurrency case.